# Data Quality Overview

This notebook summarizes the real-data feature tables used in the Thai stock DRL project. It is designed to run from the repository root on the HPC project directory, where generated `data/raw`, `data/processed`, and `reports` artifacts are available.

The notebook checks panel coverage, missingness, date ranges, feature groups, and known external-data caveats. If a generated data file is absent, the corresponding section reports it instead of failing.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import yaml

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "PROGRESS_CHECKLIST.md").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT

In [ ]:
FEATURE_CONFIGS = {
    "real_ohlcv": "config/real_ohlcv.yaml",
    "market_context": "config/real_ohlcv_market.yaml",
    "sector_context": "config/real_ohlcv_sector.yaml",
    "macro_proxy": "config/real_ohlcv_macro.yaml",
    "fundamentals": "config/real_ohlcv_fundamentals.yaml",
    "sentiment_scaffold": "config/real_ohlcv_sentiment.yaml",
    "official_macro_scaffold": "config/real_ohlcv_official_macro.yaml",
}


def load_yaml(path: str) -> dict:
    with (ROOT / path).open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def configured_feature_path(config: dict) -> Path:
    paths = config.get("paths", {})
    primary = ROOT / paths.get("features", "")
    fallback = paths.get("fallback_features_csv")
    if primary.exists():
        return primary
    if fallback and (ROOT / fallback).exists():
        return ROOT / fallback
    return primary


def load_table(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported feature table format: {path}")


configs = {name: load_yaml(path) for name, path in FEATURE_CONFIGS.items()}
feature_paths = {name: configured_feature_path(config) for name, config in configs.items()}
feature_paths

## Feature Table Availability

In [ ]:
availability = []
for name, path in feature_paths.items():
    availability.append(
        {
            "dataset": name,
            "path": str(path.relative_to(ROOT)) if path.is_absolute() else str(path),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else None,
        }
    )

availability_df = pd.DataFrame(availability)
availability_df

## Panel Coverage

In [ ]:
def panel_summary(name: str, df: pd.DataFrame) -> dict:
    dates = pd.to_datetime(df["date"])
    return {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "tickers": df["ticker"].nunique() if "ticker" in df.columns else None,
        "start_date": dates.min().date().isoformat(),
        "end_date": dates.max().date().isoformat(),
        "unique_dates": dates.nunique(),
        "duplicate_date_ticker_rows": int(df.duplicated(["date", "ticker"]).sum()) if {"date", "ticker"}.issubset(df.columns) else None,
    }


tables = {name: load_table(path) for name, path in feature_paths.items()}
loaded_tables = {name: df for name, df in tables.items() if df is not None}

if loaded_tables:
    panel_df = pd.DataFrame(panel_summary(name, df) for name, df in loaded_tables.items())
else:
    panel_df = pd.DataFrame(columns=["dataset", "rows", "columns", "tickers", "start_date", "end_date", "unique_dates", "duplicate_date_ticker_rows"])

panel_df

## Ticker-Level Coverage

In [ ]:
def ticker_coverage(name: str, df: pd.DataFrame) -> pd.DataFrame:
    if not {"date", "ticker"}.issubset(df.columns):
        return pd.DataFrame()
    work = df.copy()
    work["date"] = pd.to_datetime(work["date"])
    expected_dates = work["date"].nunique()
    rows = []
    for ticker, part in work.groupby("ticker", sort=True):
        unique_dates = part["date"].nunique()
        rows.append(
            {
                "dataset": name,
                "ticker": ticker,
                "rows": len(part),
                "start_date": part["date"].min().date().isoformat(),
                "end_date": part["date"].max().date().isoformat(),
                "unique_dates": unique_dates,
                "missing_panel_dates": expected_dates - unique_dates,
                "coverage_ratio": unique_dates / expected_dates if expected_dates else 0.0,
            }
        )
    return pd.DataFrame(rows)


coverage_df = pd.concat([ticker_coverage(name, df) for name, df in loaded_tables.items()], ignore_index=True) if loaded_tables else pd.DataFrame()
coverage_df

## Missingness By Dataset

In [ ]:
def missingness(name: str, df: pd.DataFrame) -> pd.DataFrame:
    row_count = len(df)
    return pd.DataFrame(
        {
            "dataset": name,
            "column": df.columns,
            "missing_values": [int(df[column].isna().sum()) for column in df.columns],
            "missing_ratio": [float(df[column].isna().sum() / row_count) if row_count else 0.0 for column in df.columns],
            "dtype": [str(df[column].dtype) for column in df.columns],
        }
    ).sort_values(["dataset", "missing_ratio", "column"], ascending=[True, False, True])


missing_df = pd.concat([missingness(name, df) for name, df in loaded_tables.items()], ignore_index=True) if loaded_tables else pd.DataFrame()
missing_df.head(80)

## Feature Group Presence

In [ ]:
FEATURE_GROUP_PREFIXES = {
    "technical": ["return_", "volatility_", "ma_ratio_", "rsi_", "macd", "bollinger_", "volume_ratio_", "atr_"],
    "market_index": ["set_"],
    "sector": ["sector_"],
    "macro_proxy": ["usdthb_", "brent_", "wti_", "gold_", "us10y_", "macro_rate_change"],
    "fundamental": ["fundamental_"],
    "sentiment": ["sentiment_"],
    "official_macro": ["bot_"],
}


def count_feature_groups(name: str, df: pd.DataFrame) -> dict:
    row = {"dataset": name}
    for group, prefixes in FEATURE_GROUP_PREFIXES.items():
        columns = [column for column in df.columns if any(column.startswith(prefix) for prefix in prefixes)]
        row[group] = len(columns)
    return row


group_df = pd.DataFrame(count_feature_groups(name, df) for name, df in loaded_tables.items()) if loaded_tables else pd.DataFrame()
group_df

## External Source Caveats

In [ ]:
raw_sources = {
    "real_ohlcv_prices": "data/raw/prices_real_ohlcv.csv",
    "set_index_prices": "data/raw/prices_market_indices.csv",
    "macro_proxy_prices": "data/raw/prices_macro_yahoo.csv",
    "yahoo_fundamentals": "data/raw/fundamentals_yahoo_quarterly.csv",
    "yahoo_latest_news": "data/raw/news_yahoo_latest.csv",
    "bot_official_macro_probe": "data/raw/bot_official_macro.csv",
}


def summarize_raw_source(name: str, rel_path: str) -> dict:
    path = ROOT / rel_path
    row = {"source": name, "path": rel_path, "exists": path.exists(), "rows": None, "start_date": None, "end_date": None, "columns": None}
    if not path.exists():
        return row
    df = pd.read_csv(path)
    row["rows"] = len(df)
    row["columns"] = len(df.columns)
    date_candidates = [column for column in df.columns if column.lower() in {"date", "release_date", "published_at", "period"}]
    if date_candidates:
        dates = pd.to_datetime(df[date_candidates[0]], errors="coerce")
        if dates.notna().any():
            row["start_date"] = dates.min().date().isoformat()
            row["end_date"] = dates.max().date().isoformat()
    return row


raw_summary_df = pd.DataFrame(summarize_raw_source(name, path) for name, path in raw_sources.items())
raw_summary_df

Known caveats to carry into the report:

- Yahoo Finance latest-news output is a source probe, not historical 2021-2024 news coverage.
- The BOT web-table probe validates the official macro merge path, but the collected table does not cover the 2021-2024 modeling window.
- The five-ticker sector mapping is a pilot mapping; full sector-relative features require a broader official sector/index source.
- Yahoo statement fundamentals are useful for pipeline validation but should be replaced or cross-checked with licensed SET/SETSMART fundamentals for a final research-grade run.

## Optional: Export Notebook Tables

In [ ]:
EXPORT = False
output_dir = ROOT / "reports" / "data_quality_notebook"

if EXPORT:
    output_dir.mkdir(parents=True, exist_ok=True)
    availability_df.to_csv(output_dir / "feature_table_availability.csv", index=False)
    panel_df.to_csv(output_dir / "panel_summary.csv", index=False)
    coverage_df.to_csv(output_dir / "ticker_coverage.csv", index=False)
    missing_df.to_csv(output_dir / "column_missingness.csv", index=False)
    group_df.to_csv(output_dir / "feature_group_presence.csv", index=False)
    raw_summary_df.to_csv(output_dir / "raw_source_summary.csv", index=False)

output_dir if EXPORT else "Set EXPORT = True to write CSV summaries."